# ASR Demo

This notebook installs the project, downloads the final checkpoint and KenLM model, runs a small sample, and optionally evaluates a custom dataset.

**For evaluation on your own dataset, change only `DATASET_URL` in the next cell and select `Runtime → Run all`.**

`DATASET_URL` should be a direct URL to a `.zip`, `.tar`, `.tar.gz`, or `.tgz` archive, or a public Google Drive file containing such an archive.

In [ ]:
DATASET_URL = ""  # Change only this value for the custom dataset

## 1. Clone the repository and install the locked environment

The demo is intended for a fresh Google Colab runtime. Git LFS is installed before cloning because the final checkpoint is stored with LFS.

In [ ]:
!apt-get -qq install -y git-lfs
!git lfs install

In [ ]:
!git clone -q https://github.com/antipula38/asr-homework.git
%cd asr-homework

In [ ]:
!pip -q install uv
!uv sync --locked --python 3.11

## 2. Download the final checkpoint and language model

`download_resources.py` pulls `saved/final_model/model_best.pth` through Git LFS and prepares `resources/lm/3-gram.pruned.1e-7.arpa`.

In [ ]:
!uv run --no-sync python download_resources.py


## 3. Prepare a small sample dataset

The sample follows the same `audio/` + `transcriptions/` layout required for custom evaluation.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

sample = Path("data/demo")
(sample / "audio").mkdir(parents=True, exist_ok=True)
(sample / "transcriptions").mkdir(parents=True, exist_ok=True)

urlretrieve(
    "https://download.pytorch.org/torchaudio/tutorial-assets/ctc-decoding/1688-142285-0007.wav",
    sample / "audio/1688-142285-0007.wav",
)

(sample / "transcriptions/1688-142285-0007.txt").write_text(
    "i really was very much afraid of showing him how much shocked "
    "i was at some parts of what he said\n",
    encoding="utf-8",
)

print("Sample dataset:", sample)


## 4. Run final-model inference on the sample

The demo uses the final checkpoint and `beam_lm` decoding. Predictions are saved under `outputs/demo/test/`.

In [ ]:
import os
import subprocess

os.environ["MPLBACKEND"] = "Agg"

sample_infer_cmd = [
    "uv", "run", "--no-sync", "python", "inference.py",
    "inferencer.from_pretrained=saved/final_model/model_best.pth",
    "datasets.test.data_dir=data/demo",
    "inferencer.save_path=outputs/demo",
    "dataloader.batch_size=1",
    "dataloader.num_workers=0"
]

subprocess.run(sample_infer_cmd, check=True)


In [ ]:
pred_files = sorted(Path("outputs/demo/test").glob("*.txt"))
print("Saved predictions:")
for path in pred_files:
    print(f"{path}:")
    print(path.read_text(encoding="utf-8").strip())


## 5. Calculate WER and CER on the sample

`calc_metrics.py` compares the sample ground truth and the prediction directory.

In [ ]:
subprocess.run(
    [
        "uv", "run", "--no-sync", "python", "calc_metrics.py",
        "--ground-truth", "data/demo/transcriptions",
        "--predictions", "outputs/demo/test",
    ],
    check=True,
)


## 6. Download and extract `DATASET_URL`

Expected archive structure:

```text
dataset/
├── audio/
│   ├── UtteranceID1.wav
│   ├── UtteranceID2.flac
│   └── ...
└── transcriptions/        # optional
    ├── UtteranceID1.txt
    ├── UtteranceID2.txt
    └── ...
```

The archive may contain this dataset directory at its root or one level deeper. If `DATASET_URL` is empty, the custom-dataset part is skipped.

In [ ]:
import shutil
import tarfile
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

USER_DATASET = None

if DATASET_URL.strip():

    source = Path(DATASET_URL)

    # 1. Local folder
    if source.exists() and source.is_dir():

        if not (source / "audio").is_dir():
            raise FileNotFoundError(
                f"Directory {source} does not contain audio/"
            )

        USER_DATASET = source

    # 2. Local archive
    elif source.exists() and source.is_file():

        archive = source
        extracted = Path("data/user_dataset")

        shutil.rmtree(extracted, ignore_errors=True)
        extracted.mkdir(parents=True, exist_ok=True)

        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(extracted)

        elif tarfile.is_tarfile(archive):
            with tarfile.open(archive) as tf:
                tf.extractall(extracted)

        else:
            raise ValueError("Unsupported local archive format.")

        if (extracted / "audio").is_dir():
            USER_DATASET = extracted
        else:
            audio_dirs = list(extracted.rglob("audio"))

            if not audio_dirs:
                raise FileNotFoundError(
                    "Could not find audio/ in extracted dataset."
                )

            USER_DATASET = audio_dirs[0].parent

    # 3. URL
    else:

        archive = Path("data/user_dataset_archive")
        extracted = Path("data/user_dataset")

        archive.parent.mkdir(parents=True, exist_ok=True)
        archive.unlink(missing_ok=True)

        shutil.rmtree(extracted, ignore_errors=True)
        extracted.mkdir(parents=True, exist_ok=True)

        if "drive.google.com" in DATASET_URL:
            subprocess.run(
                [
                    "uv", "run", "--with", "gdown", "gdown",
                    "--fuzzy", DATASET_URL,
                    "-O", str(archive),
                ],
                check=True,
            )
        else:
            urlretrieve(DATASET_URL, archive)

        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(extracted)

        elif tarfile.is_tarfile(archive):
            with tarfile.open(archive) as tf:
                tf.extractall(extracted)

        else:
            raise ValueError(
                "Dataset must be ZIP/TAR/TAR.GZ/TGZ."
            )

        if (extracted / "audio").is_dir():
            USER_DATASET = extracted
        else:
            audio_dirs = list(extracted.rglob("audio"))

            if not audio_dirs:
                raise FileNotFoundError(
                    "Could not find audio/ in extracted dataset."
                )

            USER_DATASET = audio_dirs[0].parent

    print("Dataset root:", USER_DATASET)
    print(
        "Audio files:",
        len(list((USER_DATASET / "audio").iterdir()))
    )

else:
    print("DATASET_URL is empty — custom dataset skipped.")

## 7. Run final-model inference on the custom dataset

Nothing in this cell needs to be edited. The dataset path is taken automatically from the previous step.

In [ ]:
if USER_DATASET is not None:
    user_infer_cmd = [
        "uv", "run", "--no-sync", "python", "inference.py",
        "inferencer.from_pretrained=saved/final_model/model_best.pth",
        f"datasets.test.data_dir={USER_DATASET}",
        "inferencer.save_path=outputs/user_dataset",
        "dataloader.batch_size=4",
        "dataloader.num_workers=0",
        "lm.enabled=true",
    ]

    subprocess.run(user_infer_cmd, check=True)

    saved = sorted(Path("outputs/user_dataset/test").glob("*.txt"))
    print(f"Saved {len(saved)} prediction file(s) to outputs/user_dataset/test/")


## 8. Calculate custom-dataset metrics

If `transcriptions/` is present, WER and CER are calculated automatically. If it is absent, inference results are still saved and metrics are skipped.

In [ ]:
if USER_DATASET is not None:
    ground_truth = USER_DATASET / "transcriptions"

    if ground_truth.is_dir():
        subprocess.run(
            [
                "uv", "run", "--no-sync", "python", "calc_metrics.py",
                "--ground-truth", str(ground_truth),
                "--predictions", "outputs/user_dataset/test",
            ],
            check=True,
        )
    else:
        print(
            "No transcriptions/ directory was found. "
            "Inference completed; WER/CER calculation was skipped."
        )


## Done